## Importando bibliotecas

In [20]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import os

## Lendo os dados

In [21]:
food_app_path = os.path.join(os.getcwd(), '..', 'data', 'logs_exp_us.csv')
food_app = pd.read_csv(food_app_path, sep='\t')

## Explorando os dados

In [22]:
# Visão geral
print(food_app.info())
print('\n', food_app.head(10))

# Estatísticas descritivas
print('\nEstatísticas descritivas:')
print('\n', food_app.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 244126 entries, 0 to 244125
Data columns (total 4 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   EventName       244126 non-null  object
 1   DeviceIDHash    244126 non-null  int64 
 2   EventTimestamp  244126 non-null  int64 
 3   ExpId           244126 non-null  int64 
dtypes: int64(3), object(1)
memory usage: 7.5+ MB
None

                  EventName         DeviceIDHash  EventTimestamp  ExpId
0         MainScreenAppear  4575588528974610257      1564029816    246
1         MainScreenAppear  7416695313311560658      1564053102    246
2  PaymentScreenSuccessful  3518123091307005509      1564054127    248
3         CartScreenAppear  3518123091307005509      1564054127    248
4  PaymentScreenSuccessful  6217807653094995999      1564055322    248
5         CartScreenAppear  6217807653094995999      1564055323    248
6       OffersScreenAppear  8351860793733343758      1564066242   

## Preparando dados para análise

### Verificação de valores duplicados e adequação dos nomes de colunas

In [23]:
# Adequação dos nomes das colunas
food_app = food_app.rename(columns={
    'EventName': 'event_name',      # nome do evento
    'DeviceIDHash': 'device_id',    # identificador de usuário exclusivo
    'EventTimestamp': 'timestamp',  # hora do evento
    'ExpId': 'exp_id'               # número do experimento: 246 e 247 são os grupos de controle, 248 é o grupo de teste
})

In [24]:
# Verificando valores duplicados
print('\n'+'Número de linhas duplicadas:')
print(food_app.duplicated().sum())


Número de linhas duplicadas:
413


In [25]:
# Visualizando as linhas duplicadas
dups_mask = food_app.duplicated(keep=False)  # exibe todas as ocorrências (original + cópia)
print(f'Linhas envolvidas em duplicações: {dups_mask.sum()}')
food_app[dups_mask].sort_values(list(food_app.columns)).head(10)

Linhas envolvidas em duplicações: 768


,event_name,device_id,timestamp,exp_id
104106,CartScreenAppear,34565258828294726,1564857221,248
104108,CartScreenAppear,34565258828294726,1564857221,248
17036,CartScreenAppear,197027893265565660,1564659614,246
17037,CartScreenAppear,197027893265565660,1564659614,246
23419,CartScreenAppear,197027893265565660,1564668928,246
23421,CartScreenAppear,197027893265565660,1564668928,246
34222,CartScreenAppear,197027893265565660,1564684544,246
34223,CartScreenAppear,197027893265565660,1564684544,246
112561,CartScreenAppear,197027893265565660,1564902904,246
112562,CartScreenAppear,197027893265565660,1564902904,246


In [26]:
# Impacto percentual dos duplicados no total do dataset
total = len(food_app)                   # total de linhas no dataset
n_dups = food_app.duplicated().sum()    # número de linhas duplicadas
pct = round(n_dups / total * 100, 2)    # percentual de linhas duplicadas
print(f'Total de linhas:     {total}')
print(f'Linhas duplicadas:   {n_dups}')
print(f'Impacto no dataset:  {pct}%')

Total de linhas:     244126
Linhas duplicadas:   413
Impacto no dataset:  0.17%


In [27]:
# Distribuição dos duplicados por tipo de evento
# Verificando se os duplicados estão concentrados em algum tipo de evento específico
dup_rows = food_app[food_app.duplicated(keep=False)]
print('Duplicatas por event_name:')
print(dup_rows.groupby('event_name').size().sort_values(ascending=False))

Duplicatas por event_name:
event_name
PaymentScreenSuccessful    348
MainScreenAppear           207
CartScreenAppear           126
Tutorial                    54
OffersScreenAppear          33
dtype: int64


In [28]:
# Duplicatas por grupo experimental (exp_id)
# Verificando se os duplicados estão concentrados em algum grupo específico
print('Duplicatas por exp_id:')
print(dup_rows.groupby('exp_id').size().sort_values(ascending=False))

Duplicatas por exp_id:
exp_id
248    310
247    233
246    225
dtype: int64


In [29]:
# Proporção de duplicatas relativa ao tamanho de cada grupo
# Necessário para entender se um grupo tem mais duplicatas proporcionalmente ao seu tamanho
total_per_group = food_app.groupby('exp_id').size().rename('total')
dups_per_group  = dup_rows.groupby('exp_id').size().rename('duplicatas')

resumo = pd.concat([total_per_group, dups_per_group], axis=1)
resumo['% duplicatas'] = (round(resumo['duplicatas'] / resumo['total'] * 100, 2).astype(str) + '%')
resumo.index.name = None  # alinhando o índice na mesma altura dos cabeçalhos

print('Proporção de duplicatas por grupo experimental:', '\n')
print(resumo.to_string())

Proporção de duplicatas por grupo experimental: 

     total  duplicatas % duplicatas
246  80304         225        0.28%
247  78075         233         0.3%
248  85747         310        0.36%


In [30]:
# Verificando se o mesmo device_id repete o evento no mesmo timestamp (duplicata verdadeira)
# Verificando se realmente é o mesmo usuário com o mesmo evento no mesmo segundo
print('Duplicatas por device_id + event_name + timestamp:')
print(food_app.duplicated(subset=['device_id', 'event_name', 'timestamp']).sum())

Duplicatas por device_id + event_name + timestamp:
413


In [31]:
# Removendo as linhas duplicadas
food_app_clean = food_app.drop_duplicates().copy()
print(f'Número de linhas após remoção de duplicatas: {len(food_app_clean)}')

Número de linhas após remoção de duplicatas: 243713


#### Conclusão de impacto 

- Como o percentual é pequeno (< 1-2%) e as duplicatas estão distribuídas uniformemente entre os grupos, o impacto é baixo e podem ser removidas com *drop_duplicates()*. 

- Caso estivessem concentradas em um grupo apenas, caberia investigar antes de remover.

### Adição de uma coluna de data e hora e uma coluna separada para datas

In [32]:
# Convertendo timestamp (int64 Unix) para datetime e extraindo a data
food_app_clean['datetime'] = pd.to_datetime(food_app_clean['timestamp'], unit='s')
food_app_clean['date']     = food_app_clean['datetime'].dt.date

print(food_app_clean[['timestamp', 'datetime', 'date']].head())
print()
print()
print(food_app_clean.info())

    timestamp            datetime        date
0  1564029816 2019-07-25 04:43:36  2019-07-25
1  1564053102 2019-07-25 11:11:42  2019-07-25
2  1564054127 2019-07-25 11:28:47  2019-07-25
3  1564054127 2019-07-25 11:28:47  2019-07-25
4  1564055322 2019-07-25 11:48:42  2019-07-25


<class 'pandas.core.frame.DataFrame'>
Index: 243713 entries, 0 to 244125
Data columns (total 6 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   event_name  243713 non-null  object        
 1   device_id   243713 non-null  int64         
 2   timestamp   243713 non-null  int64         
 3   exp_id      243713 non-null  int64         
 4   datetime    243713 non-null  datetime64[ns]
 5   date        243713 non-null  object        
dtypes: datetime64[ns](1), int64(3), object(2)
memory usage: 13.0+ MB
None


## Estudo e verificação de dados

### Quantos eventos ficam nos registros?

In [33]:
# Cada linha do DataFrame representa um evento registrado no app
# len() retorna o total de linhas — equivalente ao total de eventos após a limpeza

total_eventos = len(food_app_clean)
print(f'Total de eventos nos registros: {total_eventos:,}')

Total de eventos nos registros: 243,713



>**Raciocínio:** Após a limpeza (remoção de duplicatas), é necessário confirmar o volume real de eventos com o qual trabalharemos nas análises seguintes. Usar `len()` sobre o DataFrame limpo é a forma mais direta — cada linha representa um evento registrado.

### Quantos usuários ficam nos registros?

In [34]:
# device_id identifica cada usuário de forma única
# nunique() conta apenas os valores distintos — ignora repetições do mesmo usuário

total_usuarios = food_app_clean['device_id'].nunique()
print(f'Total de usuários nos registros: {total_usuarios:,}')

Total de usuários nos registros: 7,551



>**Raciocínio:** O `device_id` é o identificador único de cada usuário. Um mesmo usuário pode gerar vários eventos, portanto ao utilizar `nunique()` teremos apenas os valores distintos, eliminando a repetição natural de um usuário que interagiu múltiplas vezes com o app.

### Qual é o número médio de eventos por usuário?

In [35]:
# Calculando quantos eventos cada usuário gerou individualmente
eventos_por_usuario = food_app_clean.groupby('device_id').size()

# Média simples: total de eventos dividido pelo total de usuários
media = eventos_por_usuario.mean()

print(f'Média de eventos por usuário: {media:.2f}')
print()

# describe() expõe a distribuição completa — essencial para detectar
# se a média está sendo puxada por usuários extremamente ativos (outliers)
print('Distribuição do número de eventos por usuário:')
print(eventos_por_usuario.describe().round(2))

Média de eventos por usuário: 32.28

Distribuição do número de eventos por usuário:
count    7551.00
mean       32.28
std        65.15
min         1.00
25%         9.00
50%        20.00
75%        37.00
max      2307.00
dtype: float64



>**Raciocínio:** A média simples (total de eventos ÷ total de usuários) fornece uma visão geral do engajamento médio. O uso de `groupby + describe()` possibilita uma percepção da distribuição completa — a média isolada pode ser enganosa se a distribuição for muito assimétrica (ex.: poucos usuários hiper-ativos distorcendo o valor). Ver mediana, mínimo, máximo e percentis permite um diagnóstico mais honesto do comportamento da base.

### Qual é o período de tempo que os dados cobrem?

In [36]:
# Data e hora mínima e máxima registradas no dataset limpo
data_min = food_app_clean['datetime'].min()
data_max = food_app_clean['datetime'].max()

# Diferença total em dias entre o primeiro e o último evento
periodo_total = (data_max - data_min).days

print(f'Data mínima : {data_min}')
print(f'Data máxima : {data_max}')
print(f'Período total: {periodo_total} dias')

Data mínima : 2019-07-25 04:43:36
Data máxima : 2019-08-07 21:15:17
Período total: 13 dias


> **Raciocínio:** O `datetime` já foi criado na etapa de preparação; basta aplicar `.min()` e `.max()` diretamente sobre essa coluna. A subtração entre os dois valores retorna um `Timedelta` do pandas — acessamos `.days` para ter o número inteiro de dias do intervalo.

### Histograma de eventos por data e hora

In [37]:
# Extração da hora do dia para o eixo de distribuição intradiária
food_app_clean['hour'] = food_app_clean['datetime'].dt.hour     # extraindo apenas a hora (0-23) de cada registro

# --- Histograma 1: eventos por dia ---
eventos_por_dia = food_app_clean.groupby('date').size().reset_index(name='n_eventos')  # contando eventos agrupados por data
eventos_por_dia['date'] = pd.to_datetime(eventos_por_dia['date'])  # convertendo para datetime para ordenação correta no eixo x

fig_dia = px.bar(
    eventos_por_dia,
    x='date',           # eixo x: data
    y='n_eventos',      # eixo y: volume de eventos naquele dia
    title='Eventos por dia',
    labels={'date': 'Data', 'n_eventos': 'Número de eventos'}
)
fig_dia.update_layout(bargap=0.1)  # espaçamento entre barras para melhor legibilidade
fig_dia.show()

# --- Histograma 2: eventos por hora do dia ---
fig_hora = px.histogram(
    food_app_clean,
    x='hour',           # eixo x: hora do dia (0-23)
    nbins=24,           # um bin por hora — granularidade máxima sem perda de informação
    title='Distribuição de eventos por hora do dia',
    labels={'hour': 'Hora do dia', 'count': 'Número de eventos'}
)
fig_hora.update_layout(bargap=0.05, xaxis=dict(dtick=1))  # dtick=1 força exibição de todas as 24 horas no eixo x
fig_hora.show()

> **Raciocínio:** Dois histogramas complementares respondem a perguntas diferentes. O gráfico por **dia** revela se há variações de volume ao longo do período — quedas ou picos que podem sinalizar anomalias ou início de experimento. O gráfico por **hora do dia** expõe o ritmo de uso intradiário, útil para entender o comportamento do usuário e detectar se logs noturnos (horários atípicos) têm concentração anormal, o que indicaria lançamentos em lote ou artefatos técnicos. Usamos `px.bar` no primeiro (agrupamento manual por data via `groupby`) e `px.histogram` no segundo (distribuição contínua por hora).

### Os dados são igualmente completos para todo o período?

In [ ]:
# Reutilizando a contagem diária criada no bloco anterior
mediana_diaria = eventos_por_dia['n_eventos'].median()  # mediana: referência robusta, não afetada por dias atípicos

# Limiar: dias com menos de 50% da mediana são tratados como incompletos
# Esse corte é conservador — captura apenas dias visivelmente abaixo da curva
limiar = mediana_diaria * 0.5

print(f'Mediana diária de eventos : {mediana_diaria:.0f}')
print(f'Limiar de incompletude    : {limiar:.0f} eventos/dia (qualquer dia com menos eventos que isso é suspeito de estar incompleto)')
print()

dias_incompletos = eventos_por_dia[eventos_por_dia['n_eventos'] < limiar].copy()  # filtrando apenas dias abaixo do limiar

print(f'Dias com dados possivelmente incompletos: {len(dias_incompletos)}')
if not dias_incompletos.empty:  # ou seja, se houver pelo menos um dia incompleto
    print(dias_incompletos[['date', 'n_eventos']].to_string(index=False))  # exibindo sem índice numérico para leitura limpa

Mediana diária de eventos : 16563
Limiar de incompletude    : 8282 eventos/dia (qualquer dia com menos eventos que isso é suspeito de estar incompleto)

Dias com dados possivelmente incompletos: 7
      date  n_eventos
2019-07-25          9
2019-07-26         31
2019-07-27         55
2019-07-28        105
2019-07-29        184
2019-07-30        412
2019-07-31       2030


> **Raciocínio:** A mediana é preferida à média como referência porque é resistente a outliers — um dia extraordinariamente movimentado não eleva artificialmente o patamar. O limiar de 50% da mediana é deliberadamente conservador: queremos capturar apenas dias que claramente ficam fora do padrão estabelecido, não dias com pequenas variações. Dias com volume baixo no início do log tipicamente ocorrem porque o experimento ainda estava sendo rampado ou porque eventos de usuários foram registrados com atraso (retroativamente), criando uma falsa cauda inicial no dataset.

### Qual período os dados realmente representam?

In [39]:
# Definindo a data de corte: o último dia identificado como incompleto

if not dias_incompletos.empty:
    data_corte = dias_incompletos['date'].max()  # data do último dia incompleto — tudo antes é descartado
    food_app_filtered = food_app_clean[food_app_clean['datetime'] > data_corte].copy()  # mantém apenas registros após o corte
else:
    # Se não há dias incompletos, o dataset completo já é o período real
    data_corte = None
    food_app_filtered = food_app_clean.copy()

# Exibe o período real após o corte
inicio_real = food_app_filtered['date'].min()                                   # primeira data do período válido
fim_real     = food_app_filtered['date'].max()                                  # última data do período válido
periodo_real = (pd.to_datetime(fim_real) - pd.to_datetime(inicio_real)).days    # duração em dias do período limpo

print(f'Data de corte            : {data_corte}')
print(f'Período real — início    : {inicio_real}')
print(f'Período real — fim       : {fim_real}')
print(f'Duração do período real  : {periodo_real} dias')
print(f'Eventos no período real  : {len(food_app_filtered):,}')

Data de corte            : 2019-07-31 00:00:00
Período real — início    : 2019-07-31
Período real — fim       : 2019-08-07
Duração do período real  : 7 dias
Eventos no período real  : 242,917


> **Raciocínio:** Descartamos tudo que está **igual ou antes** do último dia incompleto (`>` em vez de `>=`) porque eventos do próprio dia de corte podem estar parcialmente registrados — manter a borda seria arriscado. Ao salvar o resultado em `food_app_filtered` (sem sobrescrever `food_app_clean`) preservamos o dataset limpo original para eventual reverificação. Esse filtro garante que as análises de funil e de teste A/B serão feitas sobre janelas de tempo uniformes, condição necessária para comparações justas entre os grupos de controle e experimento.

### Quantos eventos e usuários foram perdidos ao excluir os dados mais antigos?

In [41]:
# Eventos antes e depois do corte
events_before  = len(food_app_clean)                    # total de eventos no dataset limpo (antes do corte temporal)
events_after = len(food_app_filtered)                 # total de eventos após a remoção dos dias incompletos
missed_events = events_before - events_after       # diferença absoluta
pct_missed_events = missed_events / events_before * 100  # impacto percentual sobre o total

# Usuários antes e depois do corte
users_before  = food_app_clean['device_id'].nunique()     # usuários únicos antes do corte
users_after = food_app_filtered['device_id'].nunique()  # usuários únicos após o corte
missed_users = users_before - users_after        # diferença absoluta
pct_missed_users = missed_users / users_before * 100  # impacto percentual sobre o total

print('--- Impacto do corte temporal ---')
print(f'Eventos  antes : {events_before:>8,}')
print(f'Eventos  depois: {events_after:>8,}   | perdidos: {missed_events:,} ({pct_missed_events:.2f}%)')
print()
print(f'Usuários antes : {users_before:>8,}')
print(f'Usuários depois: {users_after:>8,}   | perdidos: {missed_users:,} ({pct_missed_users:.2f}%)')

--- Impacto do corte temporal ---
Eventos  antes :  243,713
Eventos  depois:  242,917   | perdidos: 796 (0.33%)

Usuários antes :    7,551
Usuários depois:    7,542   | perdidos: 9 (0.12%)


> **Raciocínio:** Comparamos as mesmas métricas (eventos e usuários únicos) antes e depois do corte para quantificar o custo da limpeza. O percentual de perda é o indicador mais importante: uma perda pequena valida a decisão de descartar os dias incompletos sem comprometer a representatividade da amostra. Uma perda grande exigiria reconsiderar a estratégia de corte.

### Os três grupos experimentais ainda estão presentes após o corte?

In [42]:
# Quantificando usuários únicos por grupo experimental no dataset filtrado
users_per_group = (
    food_app_filtered
    .groupby('exp_id')['device_id']   # agrupa por grupo e seleciona a coluna de usuário
    .nunique()                         # conta apenas IDs distintos dentro de cada grupo
    .rename('usuarios_unicos')
    .reset_index()
)

# Adiciona o percentual de cada grupo em relação ao total de usuários filtrados
users_per_group['% do total'] = (
    users_per_group['usuarios_unicos'] / users_per_group['usuarios_unicos'].sum() * 100
).round(2).astype(str) + '%'

print('Usuários únicos por grupo experimental (após corte):')
print(users_per_group.to_string(index=False))
print()

# Confirmação de que os três grupos estão presentes
grupos_esperados = {246, 247, 248}                             # grupos definidos no experimento
grupos_presentes = set(users_per_group['exp_id'].values)       # grupos encontrados no dataset filtrado
grupos_ausentes  = grupos_esperados - grupos_presentes         # diferença: grupos que deveriam estar mas não estão

if not grupos_ausentes:
    print('✓ Todos os três grupos experimentais (246, 247, 248) estão presentes.')
else:
    print(f'✗ Grupos ausentes após o corte: {grupos_ausentes}')

Usuários únicos por grupo experimental (após corte):
 exp_id  usuarios_unicos % do total
    246             2485     32.95%
    247             2517     33.37%
    248             2540     33.68%

✓ Todos os três grupos experimentais (246, 247, 248) estão presentes.


> **Raciocínio:** O corte temporal descarta eventos com base em data — não em grupo. Ainda assim, é necessário verificar se os três grupos (246 e 247 de controle, 248 de teste) sobreviveram ao filtro, pois um grupo sub-representado nos dias iniciais poderia desaparecer completamente. Além da presença, a coluna `% do total` permite detectar desequilíbrios severos entre grupos — condição que comprometeria qualquer comparação estatística posterior. A comparação com `set` é a forma mais direta de checar se todos os grupos esperados estão no resultado.